# Checkpoint 2: for room amenity processing

In [1]:
import pandas as pd
df = pd.read_csv("room_amenities_mapped.csv")
df.drop(columns=["amenity_group"], inplace=True)

In [2]:
df

,room_type_id,room_amenities
0,1,"[{'amenity': 'Diện tích phòng: 18 m²', 'type':..."
1,2,"[{'amenity': 'Diện tích phòng: 30 m²', 'type':..."
2,3,"[{'amenity': 'Diện tích phòng: 45 m²', 'type':..."
3,4,"[{'amenity': 'Diện tích phòng: 20 m²', 'type':..."
4,5,"[{'amenity': 'Diện tích phòng: 35 m²', 'type':..."
...,...,...
12042,11710,[]
12043,11711,[]
12044,11718,[]
12045,11759,[]


## Handling the bed amenity

In [3]:
import ast
import re
import pandas as pd

qty_pattern = re.compile(r"\b\d+\s*giường\b", re.IGNORECASE)

def extract_bed_features(amenities):
    # handle NaN
    if pd.isna(amenities):
        return None

    # convert string -> list
    if isinstance(amenities, str):
        try:
            amenities = ast.literal_eval(amenities)
        except:
            return None

    bed_candidates = []

    # collect all bed items
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "bed":
            val = amenity.get("amenity")
            if val:
                bed_candidates.append(str(val).strip())

    if not bed_candidates:
        return None

    # ✅ priority: bed strings with quantity
    for b in bed_candidates:
        if qty_pattern.search(b):
            return b

    # fallback: return the most informative one (longest)
    return max(bed_candidates, key=len)

In [4]:

df["bed_str"] = df["room_amenities"].apply(extract_bed_features)

In [5]:
df

,room_type_id,room_amenities,bed_str
0,1,"[{'amenity': 'Diện tích phòng: 18 m²', 'type':...",1 giường đôi lớn
1,2,"[{'amenity': 'Diện tích phòng: 30 m²', 'type':...",1 giường lớn
2,3,"[{'amenity': 'Diện tích phòng: 45 m²', 'type':...",2 giường lớn
3,4,"[{'amenity': 'Diện tích phòng: 20 m²', 'type':...",1 giường lớn
4,5,"[{'amenity': 'Diện tích phòng: 35 m²', 'type':...",1 giường đôi lớn hoặc 2 giường đơn
...,...,...,...
12042,11710,[],None
12043,11711,[],None
12044,11718,[],None
12045,11759,[],None


In [6]:
df["bed_str"] = df["bed_str"].str.replace(r"/", "&")
df["bed_str"] = df["bed_str"].str.replace(r"và", "&")

In [7]:
df["expanded_bed"] = df["bed_str"].str.split("&|hoặc").apply(lambda x: [item.strip() for item in x] if isinstance(x, list) else x)

In [8]:
import re

def process_expanded_bed(bed_list):
    if not isinstance(bed_list, list):
        return []
    processed_beds = []
    for bed in bed_list:
        if not isinstance(bed, str):
            continue
        # Tách các lựa chọn "hoặc"
        for part in re.split(r"hoặc", bed):
            part = part.strip()
            # Tìm số lượng và loại giường
            match = re.match(r"(\d+)\s*(.*)", part)
            if match:
                bed_count = int(match.group(1))
                bed_type = match.group(2).strip()
            else:
                bed_count = 1
                bed_type = part
            if bed_type:
                processed_beds.append({'bed_type': bed_type, 'bed_count': bed_count})
    return processed_beds

In [9]:
df["expanded_bed"] = df["expanded_bed"].apply(process_expanded_bed)

In [10]:
df

,room_type_id,room_amenities,bed_str,expanded_bed
0,1,"[{'amenity': 'Diện tích phòng: 18 m²', 'type':...",1 giường đôi lớn,"[{'bed_type': 'giường đôi lớn', 'bed_count': 1}]"
1,2,"[{'amenity': 'Diện tích phòng: 30 m²', 'type':...",1 giường lớn,"[{'bed_type': 'giường lớn', 'bed_count': 1}]"
2,3,"[{'amenity': 'Diện tích phòng: 45 m²', 'type':...",2 giường lớn,"[{'bed_type': 'giường lớn', 'bed_count': 2}]"
3,4,"[{'amenity': 'Diện tích phòng: 20 m²', 'type':...",1 giường lớn,"[{'bed_type': 'giường lớn', 'bed_count': 1}]"
4,5,"[{'amenity': 'Diện tích phòng: 35 m²', 'type':...",1 giường đôi lớn hoặc 2 giường đơn,"[{'bed_type': 'giường đôi lớn', 'bed_count': 1..."
...,...,...,...,...
12042,11710,[],None,[]
12043,11711,[],None,[]
12044,11718,[],None,[]
12045,11759,[],None,[]


In [11]:
# df.to_csv(
#     "checkingbed.csv",
#     index=False,
#     encoding="utf-8-sig"   # để mở bằng Excel không lỗi dấu
# )

In [12]:
import ast
import pandas as pd
import numpy as np

def parse_expanded_bed(x):
    # ✅ Nếu x là NaN (float) thì return []
    if x is None:
        return []
    if isinstance(x, float) and pd.isna(x):
        return []

    # ✅ Nếu x là string JSON => convert thành python object
    if isinstance(x, str):
        x = x.strip()
        if x == "" or x.lower() == "nan":
            return []
        try:
            x = ast.literal_eval(x)
        except:
            return []

    beds = []

    # ✅ Nếu x là list/tuple/np.array
    if isinstance(x, (list, tuple, np.ndarray)):
        for item in x:
            if isinstance(item, dict):
                bt = item.get("bed_type") or item.get("type") or item.get("name")
                if bt:
                    beds.append(str(bt).strip())
            elif isinstance(item, str):
                beds.append(item.strip())

    # ✅ Nếu x là dict
    elif isinstance(x, dict):
        bt = x.get("bed_type") or x.get("type") or x.get("name")
        if bt:
            beds.append(str(bt).strip())

    return beds


# Parse ra list giường
df["bed_list"] = df["expanded_bed"].apply(parse_expanded_bed)

# explode ra từng giường
beds_series = df["bed_list"].explode().dropna()

print("Tổng số loại giường unique:", beds_series.nunique())
print("Danh sách loại giường:", beds_series.drop_duplicates().tolist())


Tổng số loại giường unique: 10
Danh sách loại giường: ['giường đôi lớn', 'giường lớn', 'giường đơn', 'giường sofa', 'giường đôi', 'giường đôi nhỏ', 'giường siêu lớn', 'nệm futon', 'giường tầng', 'Giường cực dài']


In [13]:
bed_values = {
    'giường đôi lớn': "large_double_bed",
    'giường lớn': "large_bed",
    'giường đơn' : "single_bed",
    'giường sofa': "sofa_bed",
    'giường đôi' : "double_bed",
    'giường đôi nhỏ': "small_double_bed",
    'giường siêu lớn': "king_size_bed",
    'nệm futon': "futon_mattress",
    'giường tầng': "bunk_bed",
    'Giường cực dài': "extra_long_bed"
}

In [14]:
def extract_bed_large_double_bed(expanded_bed):
    for bed in expanded_bed:
        if bed['bed_type'] == "giường đôi lớn":
            return 1
    return 0

df["large_double_bed"] = df["expanded_bed"].apply(extract_bed_large_double_bed)

In [15]:
def extract_bed_large_bed(expanded_bed):
    for bed in expanded_bed:
        if bed['bed_type'] == "giường lớn":
            return 1
    return 0
df["large_bed"] = df["expanded_bed"].apply(extract_bed_large_bed)

In [16]:
df

,room_type_id,room_amenities,bed_str,expanded_bed,bed_list,large_double_bed,large_bed
0,1,"[{'amenity': 'Diện tích phòng: 18 m²', 'type':...",1 giường đôi lớn,"[{'bed_type': 'giường đôi lớn', 'bed_count': 1}]",[giường đôi lớn],1,0
1,2,"[{'amenity': 'Diện tích phòng: 30 m²', 'type':...",1 giường lớn,"[{'bed_type': 'giường lớn', 'bed_count': 1}]",[giường lớn],0,1
2,3,"[{'amenity': 'Diện tích phòng: 45 m²', 'type':...",2 giường lớn,"[{'bed_type': 'giường lớn', 'bed_count': 2}]",[giường lớn],0,1
3,4,"[{'amenity': 'Diện tích phòng: 20 m²', 'type':...",1 giường lớn,"[{'bed_type': 'giường lớn', 'bed_count': 1}]",[giường lớn],0,1
4,5,"[{'amenity': 'Diện tích phòng: 35 m²', 'type':...",1 giường đôi lớn hoặc 2 giường đơn,"[{'bed_type': 'giường đôi lớn', 'bed_count': 1...","[giường đôi lớn, giường đơn]",1,0
...,...,...,...,...,...,...,...
12042,11710,[],None,[],[],0,0
12043,11711,[],None,[],[],0,0
12044,11718,[],None,[],[],0,0
12045,11759,[],None,[],[],0,0


In [17]:
def extract_bed_single_bed(expanded_bed):
    for bed in expanded_bed:
        if bed['bed_type'] == "giường đơn":
            return 1
    return 0

df["single_bed"] = df["expanded_bed"].apply(extract_bed_single_bed)

In [18]:
def extract_bed_sofa_bed(expanded_bed):
    for bed in expanded_bed:
        if bed['bed_type'] == "giường sofa":
            return 1
    return 0

df["sofa_bed"] = df["expanded_bed"].apply(extract_bed_sofa_bed)

In [19]:
def extract_bed_double_bed(expanded_bed):
    for bed in expanded_bed:
        if bed['bed_type'] == "giường đôi":
            return 1
    return 0

df["double_bed"] = df["expanded_bed"].apply(extract_bed_double_bed)

In [20]:
def extract_bed_small_double_bed(expanded_bed):
    res = 0
    for bed in expanded_bed:
        if bed['bed_type'] == "giường đôi nhỏ":
            return 1
    return 0

df["small_double_bed"] = df["expanded_bed"].apply(extract_bed_small_double_bed)

In [21]:
def extract_bed_king_size_bed(expanded_bed):
    res = 0
    for bed in expanded_bed:
        if bed['bed_type'] == "giường siêu lớn":
            return 1
    return 0

df["king_size_bed"] = df["expanded_bed"].apply(extract_bed_king_size_bed)

In [22]:
def extract_bed_futon_mattress(expanded_bed):
    res = 0
    for bed in expanded_bed:
        if bed['bed_type'] == "nệm futon":
            return 1
    return 0

df["futon_mattress"] = df["expanded_bed"].apply(extract_bed_futon_mattress)

In [23]:
def extract_bed_bunk_bed(expanded_bed):
    res = 0
    for bed in expanded_bed:
        if bed['bed_type'] == "giường tầng":
            return 1
    return 0

df["bunk_bed"] = df["expanded_bed"].apply(extract_bed_bunk_bed)

In [24]:
# def extract_bed_extra_long_bed(expanded_bed):
#     res = 0
#     for bed in expanded_bed:
#         if bed['bed_type'] == "Giường cực dài":
#             return 1
#     return 0

# df["extra_long_bed"] = df["expanded_bed"].apply(extract_bed_extra_long_bed)

In [25]:
df

,room_type_id,room_amenities,bed_str,expanded_bed,bed_list,large_double_bed,large_bed,single_bed,sofa_bed,double_bed,small_double_bed,king_size_bed,futon_mattress,bunk_bed
0,1,"[{'amenity': 'Diện tích phòng: 18 m²', 'type':...",1 giường đôi lớn,"[{'bed_type': 'giường đôi lớn', 'bed_count': 1}]",[giường đôi lớn],1,0,0,0,0,0,0,0,0
1,2,"[{'amenity': 'Diện tích phòng: 30 m²', 'type':...",1 giường lớn,"[{'bed_type': 'giường lớn', 'bed_count': 1}]",[giường lớn],0,1,0,0,0,0,0,0,0
2,3,"[{'amenity': 'Diện tích phòng: 45 m²', 'type':...",2 giường lớn,"[{'bed_type': 'giường lớn', 'bed_count': 2}]",[giường lớn],0,1,0,0,0,0,0,0,0
3,4,"[{'amenity': 'Diện tích phòng: 20 m²', 'type':...",1 giường lớn,"[{'bed_type': 'giường lớn', 'bed_count': 1}]",[giường lớn],0,1,0,0,0,0,0,0,0
4,5,"[{'amenity': 'Diện tích phòng: 35 m²', 'type':...",1 giường đôi lớn hoặc 2 giường đơn,"[{'bed_type': 'giường đôi lớn', 'bed_count': 1...","[giường đôi lớn, giường đơn]",1,0,1,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12042,11710,[],None,[],[],0,0,0,0,0,0,0,0,0
12043,11711,[],None,[],[],0,0,0,0,0,0,0,0,0
12044,11718,[],None,[],[],0,0,0,0,0,0,0,0,0
12045,11759,[],None,[],[],0,0,0,0,0,0,0,0,0


In [26]:
# df.to_csv("checkpoint3.csv", index=False)

In [27]:
def calculate_flexibility_score(bed_str):
    # đếm số lần xuất hiện từ "hoặc"
    return bed_str.count("hoặc") if isinstance(bed_str, str) else 0
df["flexibility_score"] = df["bed_str"].apply(calculate_flexibility_score) + 1

In [28]:
df[["room_type_id","room_amenities", "flexibility_score"]]

,room_type_id,room_amenities,flexibility_score
0,1,"[{'amenity': 'Diện tích phòng: 18 m²', 'type':...",1
1,2,"[{'amenity': 'Diện tích phòng: 30 m²', 'type':...",1
2,3,"[{'amenity': 'Diện tích phòng: 45 m²', 'type':...",1
3,4,"[{'amenity': 'Diện tích phòng: 20 m²', 'type':...",1
4,5,"[{'amenity': 'Diện tích phòng: 35 m²', 'type':...",2
...,...,...,...
12042,11710,[],1
12043,11711,[],1
12044,11718,[],1
12045,11759,[],1


In [29]:
df.drop(columns=["expanded_bed", "bed_str"])

,room_type_id,room_amenities,bed_list,large_double_bed,large_bed,single_bed,sofa_bed,double_bed,small_double_bed,king_size_bed,futon_mattress,bunk_bed,flexibility_score
0,1,"[{'amenity': 'Diện tích phòng: 18 m²', 'type':...",[giường đôi lớn],1,0,0,0,0,0,0,0,0,1
1,2,"[{'amenity': 'Diện tích phòng: 30 m²', 'type':...",[giường lớn],0,1,0,0,0,0,0,0,0,1
2,3,"[{'amenity': 'Diện tích phòng: 45 m²', 'type':...",[giường lớn],0,1,0,0,0,0,0,0,0,1
3,4,"[{'amenity': 'Diện tích phòng: 20 m²', 'type':...",[giường lớn],0,1,0,0,0,0,0,0,0,1
4,5,"[{'amenity': 'Diện tích phòng: 35 m²', 'type':...","[giường đôi lớn, giường đơn]",1,0,1,0,0,0,0,0,0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...
12042,11710,[],[],0,0,0,0,0,0,0,0,0,1
12043,11711,[],[],0,0,0,0,0,0,0,0,0,1
12044,11718,[],[],0,0,0,0,0,0,0,0,0,1
12045,11759,[],[],0,0,0,0,0,0,0,0,0,1


In [30]:
# df.to_csv("checkpoint4.csv", index=False)

In [31]:
df

,room_type_id,room_amenities,bed_str,expanded_bed,bed_list,large_double_bed,large_bed,single_bed,sofa_bed,double_bed,small_double_bed,king_size_bed,futon_mattress,bunk_bed,flexibility_score
0,1,"[{'amenity': 'Diện tích phòng: 18 m²', 'type':...",1 giường đôi lớn,"[{'bed_type': 'giường đôi lớn', 'bed_count': 1}]",[giường đôi lớn],1,0,0,0,0,0,0,0,0,1
1,2,"[{'amenity': 'Diện tích phòng: 30 m²', 'type':...",1 giường lớn,"[{'bed_type': 'giường lớn', 'bed_count': 1}]",[giường lớn],0,1,0,0,0,0,0,0,0,1
2,3,"[{'amenity': 'Diện tích phòng: 45 m²', 'type':...",2 giường lớn,"[{'bed_type': 'giường lớn', 'bed_count': 2}]",[giường lớn],0,1,0,0,0,0,0,0,0,1
3,4,"[{'amenity': 'Diện tích phòng: 20 m²', 'type':...",1 giường lớn,"[{'bed_type': 'giường lớn', 'bed_count': 1}]",[giường lớn],0,1,0,0,0,0,0,0,0,1
4,5,"[{'amenity': 'Diện tích phòng: 35 m²', 'type':...",1 giường đôi lớn hoặc 2 giường đơn,"[{'bed_type': 'giường đôi lớn', 'bed_count': 1...","[giường đôi lớn, giường đơn]",1,0,1,0,0,0,0,0,0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12042,11710,[],None,[],[],0,0,0,0,0,0,0,0,0,1
12043,11711,[],None,[],[],0,0,0,0,0,0,0,0,0,1
12044,11718,[],None,[],[],0,0,0,0,0,0,0,0,0,1
12045,11759,[],None,[],[],0,0,0,0,0,0,0,0,0,1


## Handling the sqm (area)

In [32]:
import ast
def extract_sqm_features(amenities):
    if isinstance(amenities, str):
        amenities = ast.literal_eval(amenities)
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "sqm":
           return amenity.get("amenity")
    return None

df["sqm_str"] = df["room_amenities"].apply(extract_sqm_features)

In [33]:
df

,room_type_id,room_amenities,bed_str,expanded_bed,bed_list,large_double_bed,large_bed,single_bed,sofa_bed,double_bed,small_double_bed,king_size_bed,futon_mattress,bunk_bed,flexibility_score,sqm_str
0,1,"[{'amenity': 'Diện tích phòng: 18 m²', 'type':...",1 giường đôi lớn,"[{'bed_type': 'giường đôi lớn', 'bed_count': 1}]",[giường đôi lớn],1,0,0,0,0,0,0,0,0,1,Diện tích phòng: 18 m²
1,2,"[{'amenity': 'Diện tích phòng: 30 m²', 'type':...",1 giường lớn,"[{'bed_type': 'giường lớn', 'bed_count': 1}]",[giường lớn],0,1,0,0,0,0,0,0,0,1,Diện tích phòng: 30 m²
2,3,"[{'amenity': 'Diện tích phòng: 45 m²', 'type':...",2 giường lớn,"[{'bed_type': 'giường lớn', 'bed_count': 2}]",[giường lớn],0,1,0,0,0,0,0,0,0,1,Diện tích phòng: 45 m²
3,4,"[{'amenity': 'Diện tích phòng: 20 m²', 'type':...",1 giường lớn,"[{'bed_type': 'giường lớn', 'bed_count': 1}]",[giường lớn],0,1,0,0,0,0,0,0,0,1,Diện tích phòng: 20 m²
4,5,"[{'amenity': 'Diện tích phòng: 35 m²', 'type':...",1 giường đôi lớn hoặc 2 giường đơn,"[{'bed_type': 'giường đôi lớn', 'bed_count': 1...","[giường đôi lớn, giường đơn]",1,0,1,0,0,0,0,0,0,2,Diện tích phòng: 35 m²
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12042,11710,[],None,[],[],0,0,0,0,0,0,0,0,0,1,None
12043,11711,[],None,[],[],0,0,0,0,0,0,0,0,0,1,None
12044,11718,[],None,[],[],0,0,0,0,0,0,0,0,0,1,None
12045,11759,[],None,[],[],0,0,0,0,0,0,0,0,0,1,None


In [34]:
import re
def extract_room_area(text):
    """
    Trích xuất diện tích phòng (số nguyên, đơn vị m²) từ chuỗi.
    Ví dụ: "25 m²" hoặc "Diện tích phòng: 20 m²" -> 25 hoặc 20
    """
    if not isinstance(text, str):
        return None
    match = re.search(r'(\d+)\s*m²', text)
    if match:
        return int(match.group(1))
    return None

In [35]:
df["sqm"] = df["sqm_str"].apply(extract_room_area)

In [36]:
df.drop(columns=["sqm_str"], inplace=True)

In [37]:
df

,room_type_id,room_amenities,bed_str,expanded_bed,bed_list,large_double_bed,large_bed,single_bed,sofa_bed,double_bed,small_double_bed,king_size_bed,futon_mattress,bunk_bed,flexibility_score,sqm
0,1,"[{'amenity': 'Diện tích phòng: 18 m²', 'type':...",1 giường đôi lớn,"[{'bed_type': 'giường đôi lớn', 'bed_count': 1}]",[giường đôi lớn],1,0,0,0,0,0,0,0,0,1,18.0
1,2,"[{'amenity': 'Diện tích phòng: 30 m²', 'type':...",1 giường lớn,"[{'bed_type': 'giường lớn', 'bed_count': 1}]",[giường lớn],0,1,0,0,0,0,0,0,0,1,30.0
2,3,"[{'amenity': 'Diện tích phòng: 45 m²', 'type':...",2 giường lớn,"[{'bed_type': 'giường lớn', 'bed_count': 2}]",[giường lớn],0,1,0,0,0,0,0,0,0,1,45.0
3,4,"[{'amenity': 'Diện tích phòng: 20 m²', 'type':...",1 giường lớn,"[{'bed_type': 'giường lớn', 'bed_count': 1}]",[giường lớn],0,1,0,0,0,0,0,0,0,1,20.0
4,5,"[{'amenity': 'Diện tích phòng: 35 m²', 'type':...",1 giường đôi lớn hoặc 2 giường đơn,"[{'bed_type': 'giường đôi lớn', 'bed_count': 1...","[giường đôi lớn, giường đơn]",1,0,1,0,0,0,0,0,0,2,35.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12042,11710,[],None,[],[],0,0,0,0,0,0,0,0,0,1,NaN
12043,11711,[],None,[],[],0,0,0,0,0,0,0,0,0,1,NaN
12044,11718,[],None,[],[],0,0,0,0,0,0,0,0,0,1,NaN
12045,11759,[],None,[],[],0,0,0,0,0,0,0,0,0,1,NaN


In [38]:
# df.to_csv("checkpoint5.csv", index=False)

## Handling bathroom

In [39]:
import ast
def extract_bathrooms_features(amenities):
    if isinstance(amenities, str):
        amenities = ast.literal_eval(amenities)
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "bathrooms":
           return amenity.get("amenity")
    return None

df["bathrooms_str"] = df["room_amenities"].apply(extract_bathrooms_features)

In [40]:
df[["room_type_id","room_amenities", "bathrooms_str"]]

,room_type_id,room_amenities,bathrooms_str
0,1,"[{'amenity': 'Diện tích phòng: 18 m²', 'type':...",phòng tắm riêng
1,2,"[{'amenity': 'Diện tích phòng: 30 m²', 'type':...",phòng tắm riêng
2,3,"[{'amenity': 'Diện tích phòng: 45 m²', 'type':...",None
3,4,"[{'amenity': 'Diện tích phòng: 20 m²', 'type':...",phòng tắm riêng
4,5,"[{'amenity': 'Diện tích phòng: 35 m²', 'type':...",phòng tắm riêng
...,...,...,...
12042,11710,[],None
12043,11711,[],None
12044,11718,[],None
12045,11759,[],None


In [41]:
df[["bathrooms_str"]].drop_duplicates()

,bathrooms_str
0,phòng tắm riêng
2,None
9,2 phòng tắm
66,3 phòng tắm
84,Phòng tắm chung
165,12 phòng tắm
266,1 phòng tắm
337,5 phòng tắm
533,10 phòng tắm
594,7 phòng tắm


In [42]:
import re
import pandas as pd

def process_bathroom_info(text):
    if not isinstance(text, str):
        # Trường hợp None, giả sử là 0 phòng tắm và không xác định loại
        return pd.Series([0, 1]) 
    
    text_lower = text.lower()
    
    # 1. Xử lý số lượng phòng tắm
    count = 0
    # Tìm số trong chuỗi (ví dụ: "2 phòng tắm")
    match = re.search(r'(\d+)\s*phòng tắm', text_lower)
    if match:
        count = int(match.group(1))
    # Các trường hợp mô tả chữ -> gán là 1
    elif "phòng tắm" in text_lower: 
        count = 1
        
    # 2. Xử lý loại phòng tắm (Riêng tư hay Chung)
    # Mặc định là riêng tư (1), nếu thấy chữ "chung" thì là (0)
    is_private = 1
    if "chung" in text_lower:
        is_private = 0
        
    return pd.Series([count, is_private])

# Áp dụng vào DataFrame
df[["bathroom_count", "is_private_bathroom"]] = df["bathrooms_str"].apply(process_bathroom_info)

In [43]:

# Kiểm tra kết quả
df[["bathrooms_str", "bathroom_count", "is_private_bathroom"]].drop_duplicates()

,bathrooms_str,bathroom_count,is_private_bathroom
0,phòng tắm riêng,1,1
2,None,0,1
9,2 phòng tắm,2,1
66,3 phòng tắm,3,1
84,Phòng tắm chung,1,0
165,12 phòng tắm,12,1
266,1 phòng tắm,1,1
337,5 phòng tắm,5,1
533,10 phòng tắm,10,1
594,7 phòng tắm,7,1


In [44]:
df.drop(columns=["bathrooms_str"], inplace=True)

In [45]:
df

,room_type_id,room_amenities,bed_str,expanded_bed,bed_list,large_double_bed,large_bed,single_bed,sofa_bed,double_bed,small_double_bed,king_size_bed,futon_mattress,bunk_bed,flexibility_score,sqm,bathroom_count,is_private_bathroom
0,1,"[{'amenity': 'Diện tích phòng: 18 m²', 'type':...",1 giường đôi lớn,"[{'bed_type': 'giường đôi lớn', 'bed_count': 1}]",[giường đôi lớn],1,0,0,0,0,0,0,0,0,1,18.0,1,1
1,2,"[{'amenity': 'Diện tích phòng: 30 m²', 'type':...",1 giường lớn,"[{'bed_type': 'giường lớn', 'bed_count': 1}]",[giường lớn],0,1,0,0,0,0,0,0,0,1,30.0,1,1
2,3,"[{'amenity': 'Diện tích phòng: 45 m²', 'type':...",2 giường lớn,"[{'bed_type': 'giường lớn', 'bed_count': 2}]",[giường lớn],0,1,0,0,0,0,0,0,0,1,45.0,0,1
3,4,"[{'amenity': 'Diện tích phòng: 20 m²', 'type':...",1 giường lớn,"[{'bed_type': 'giường lớn', 'bed_count': 1}]",[giường lớn],0,1,0,0,0,0,0,0,0,1,20.0,1,1
4,5,"[{'amenity': 'Diện tích phòng: 35 m²', 'type':...",1 giường đôi lớn hoặc 2 giường đơn,"[{'bed_type': 'giường đôi lớn', 'bed_count': 1...","[giường đôi lớn, giường đơn]",1,0,1,0,0,0,0,0,0,2,35.0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12042,11710,[],None,[],[],0,0,0,0,0,0,0,0,0,1,NaN,0,1
12043,11711,[],None,[],[],0,0,0,0,0,0,0,0,0,1,NaN,0,1
12044,11718,[],None,[],[],0,0,0,0,0,0,0,0,0,1,NaN,0,1
12045,11759,[],None,[],[],0,0,0,0,0,0,0,0,0,1,NaN,0,1


In [46]:
# df.to_csv("checkpoint6.csv", index=False)

## Handling views

In [47]:
import ast
def extract_views_features(amenities):
    if isinstance(amenities, str):
        amenities = ast.literal_eval(amenities)
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "views":
           return amenity.get("amenity")
    return None

df["views_str"] = df["room_amenities"].apply(extract_views_features)
df[["views_str"]]

,views_str
0,Hướng Ngoài trời
1,Hướng Núi
2,Hướng Núi
3,Hướng Ngoài trời
4,Hướng Thành phố
...,...
12042,None
12043,None
12044,None
12045,None


In [48]:
df[["views_str"]].drop_duplicates()

,views_str
0,Hướng Ngoài trời
1,Hướng Núi
4,Hướng Thành phố
10,Hướng Không có cửa sổ
11,Hướng Đường phố
12,None
29,Hướng Công viên
45,Hướng Bể bơi
52,Hướng Nông thôn
54,Hướng Vườn


In [49]:
def normalize_views(views_str):
    import unicodedata
    if isinstance(views_str, str):
        # Xóa "Hướng" và strip
        s = views_str.replace("Hướng", "").strip()
        # Chuyển sang không dấu
        s = unicodedata.normalize('NFKD', s)
        s = ''.join([c for c in s if not unicodedata.combining(c)])
        s = s.replace('Đ', 'D').replace('đ', 'd')
        return s.lower()
    return None

In [50]:
df[["views_str"]].drop_duplicates()["views_str"].apply(normalize_views)

0                          ngoai troi
1                                 nui
4                           thanh pho
10                    khong co cua so
11                          duong pho
12                               None
29                          cong vien
45                             be boi
52                          nong thon
54                               vuon
76                          san trong
160                       thien nhien
161                     ho (mot phan)
162                              song
373                              bien
374                          bai bien
377                         dai duong
379             bien (huong mot phan)
382              dai duong (mot phan)
384                              cang
437                              vinh
708                                ho
1426                       thung lung
1718                       thang canh
1931                         canh dem
7158                          dam pha
8180        

In [51]:
df["views"] = df["views_str"].apply(normalize_views)

## Other amenity

In [52]:
import ast
def extract_balcony_terrace_features(amenities):
    if isinstance(amenities, str):
        amenities = ast.literal_eval(amenities)
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "balcony-terrace":
           return True
    return None

df["balcony-terrace"] = df["room_amenities"].apply(extract_balcony_terrace_features)

In [53]:
import ast
def extract_non_smoking_room_features(amenities):
    if isinstance(amenities, str):
        amenities = ast.literal_eval(amenities)
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "non-smoking-room":
           return True
    return None

df["non-smoking-room"] = df["room_amenities"].apply(extract_non_smoking_room_features)

In [54]:
import ast
def extract_closet_features(amenities):
    if isinstance(amenities, str):
        amenities = ast.literal_eval(amenities)
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "closet":
           return True
    return None

df["closet"] = df["room_amenities"].apply(extract_closet_features)

In [55]:
import ast
def extract_air_conditioning_features(amenities):
    if isinstance(amenities, str):
        amenities = ast.literal_eval(amenities)
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "air-conditioning":
           return True
    return None

df["air_conditioning"] = df["room_amenities"].apply(extract_air_conditioning_features)

In [56]:
import ast
def extract_mini_bar_features(amenities):
    if isinstance(amenities, str):
        amenities = ast.literal_eval(amenities)
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "mini-bar":
           return True
    return None

df["mini_bar"] = df["room_amenities"].apply(extract_mini_bar_features)

In [57]:
import ast
def extract_hair_dryer_features(amenities):
    if isinstance(amenities, str):
        amenities = ast.literal_eval(amenities)
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "hair-dryer":
           return True
    return None

df["hair_dryer"] = df["room_amenities"].apply(extract_hair_dryer_features)

In [58]:
# import ast
# def extract_extra_long_beds_features(amenities):
#     if isinstance(amenities, str):
#         amenities = ast.literal_eval(amenities)
#     for amenity in amenities:
#         if isinstance(amenity, dict) and amenity.get("type") == "extra-long-beds":
#            return True
#     return None

# df["extra-long-beds"] = df["room_amenities"].apply(extract_extra_long_beds_features)

In [59]:
import ast
def extract_complimentary_bottled_water_features(amenities):
    if isinstance(amenities, str):
        amenities = ast.literal_eval(amenities)
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "complimentary-bottled-water":
           return True
    return None

df["complimentary-bottled-water"] = df["room_amenities"].apply(extract_complimentary_bottled_water_features)

In [60]:
import ast
def extract_bathtub_features(amenities):
    if isinstance(amenities, str):
        amenities = ast.literal_eval(amenities)
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "bathtub":
           return True
    return None

df["bathtub"] = df["room_amenities"].apply(extract_bathtub_features)

In [61]:
import ast
def extract_features(amenities):
    if isinstance(amenities, str):
        amenities = ast.literal_eval(amenities)
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "shower":
           return True
    return None

df["shower"] = df["room_amenities"].apply(extract_features)

In [62]:
import ast
def extract_features(amenities):
    if isinstance(amenities, str):
        amenities = ast.literal_eval(amenities)
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "separate-shower-and-tub":
           return True
    return None

df["separate-shower-and-tub"] = df["room_amenities"].apply(extract_features)

In [63]:
import ast
def extract_features(amenities):
    if isinstance(amenities, str):
        amenities = ast.literal_eval(amenities)
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "refrigerator":
           return True
    return None

df["refrigerator"] = df["room_amenities"].apply(extract_features)

In [64]:
import ast
def extract_features(amenities):
    if isinstance(amenities, str):
        amenities = ast.literal_eval(amenities)
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "high-floor":
           return True
    return None

df["high-floor"] = df["room_amenities"].apply(extract_features)

In [65]:
import ast
def extract_features(amenities):
    if isinstance(amenities, str):
        amenities = ast.literal_eval(amenities)
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "dressing-room":
           return True
    return None

df["dressing-room"] = df["room_amenities"].apply(extract_features)

In [66]:
import ast
def extract_features(amenities):
    if isinstance(amenities, str):
        amenities = ast.literal_eval(amenities)
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "ground-floor":
           return True
    return None

df["ground-floor"] = df["room_amenities"].apply(extract_features)

In [67]:
import ast
def extract_features(amenities):
    if isinstance(amenities, str):
        amenities = ast.literal_eval(amenities)
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "private-pool":
           return True
    return None

df["private-pool"] = df["room_amenities"].apply(extract_features)

In [68]:
import ast
def extract_features(amenities):
    if isinstance(amenities, str):
        amenities = ast.literal_eval(amenities)
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "executive-lounge-access":
           return True
    return None

df["executive-lounge-access"] = df["room_amenities"].apply(extract_features)

In [69]:
import ast
def extract_features(amenities):
    if isinstance(amenities, str):
        amenities = ast.literal_eval(amenities)
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "top-floor":
           return True
    return None

df["top-floor"] = df["room_amenities"].apply(extract_features)

In [70]:
import ast
def extract_features(amenities):
    if isinstance(amenities, str):
        amenities = ast.literal_eval(amenities)
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "complimentary-instant-coffee":
           return True
    return None

df["complimentary-instant-coffee"] = df["room_amenities"].apply(extract_features)

In [71]:
import ast
def extract_features(amenities):
    if isinstance(amenities, str):
        amenities = ast.literal_eval(amenities)
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "jacuzzi-bathtub":
           return True
    return None

df["jacuzzi-bathtub"] = df["room_amenities"].apply(extract_features)

In [72]:
import ast
def extract_features(amenities):
    if isinstance(amenities, str):
        amenities = ast.literal_eval(amenities)
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "smoking-allowed":
           return True
    return None

df["smoking-allowed"] = df["room_amenities"].apply(extract_features)

In [73]:
import ast
def extract_features(amenities):
    if isinstance(amenities, str):
        amenities = ast.literal_eval(amenities)
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "electric-blanket":
           return True
    return None

df["electric-blanket"] = df["room_amenities"].apply(extract_features)

In [74]:
import ast
def extract_features(amenities):
    if isinstance(amenities, str):
        amenities = ast.literal_eval(amenities)
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "free-welcome-drink":
           return True
    return None

df["free-welcome-drink"] = df["room_amenities"].apply(extract_features)

In [75]:
import ast
def extract_features(amenities):
    if isinstance(amenities, str):
        amenities = ast.literal_eval(amenities)
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "low-floor":
           return True
    return None

df["low-floor"] = df["room_amenities"].apply(extract_features)

In [76]:
import ast
def extract_features(amenities):
    if isinstance(amenities, str):
        amenities = ast.literal_eval(amenities)
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "complimentary-tea":
           return True
    return None

df["complimentary-tea"] = df["room_amenities"].apply(extract_features)

In [77]:
import ast
def extract_features(amenities):
    if isinstance(amenities, str):
        amenities = ast.literal_eval(amenities)
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "coffee-tea-maker":
           return True
    return None

df["coffee-tea-maker"] = df["room_amenities"].apply(extract_features)

In [78]:
import ast
def extract_features(amenities):
    if isinstance(amenities, str):
        amenities = ast.literal_eval(amenities)
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "hot-spring-access":
           return True
    return None

df["hot-spring-access"] = df["room_amenities"].apply(extract_features)

In [79]:
df

,room_type_id,room_amenities,bed_str,expanded_bed,bed_list,large_double_bed,large_bed,single_bed,sofa_bed,double_bed,...,top-floor,complimentary-instant-coffee,jacuzzi-bathtub,smoking-allowed,electric-blanket,free-welcome-drink,low-floor,complimentary-tea,coffee-tea-maker,hot-spring-access
0,1,"[{'amenity': 'Diện tích phòng: 18 m²', 'type':...",1 giường đôi lớn,"[{'bed_type': 'giường đôi lớn', 'bed_count': 1}]",[giường đôi lớn],1,0,0,0,0,...,None,None,None,None,None,None,None,None,None,None
1,2,"[{'amenity': 'Diện tích phòng: 30 m²', 'type':...",1 giường lớn,"[{'bed_type': 'giường lớn', 'bed_count': 1}]",[giường lớn],0,1,0,0,0,...,None,None,None,None,None,None,True,None,None,None
2,3,"[{'amenity': 'Diện tích phòng: 45 m²', 'type':...",2 giường lớn,"[{'bed_type': 'giường lớn', 'bed_count': 2}]",[giường lớn],0,1,0,0,0,...,None,None,None,True,None,None,True,None,None,None
3,4,"[{'amenity': 'Diện tích phòng: 20 m²', 'type':...",1 giường lớn,"[{'bed_type': 'giường lớn', 'bed_count': 1}]",[giường lớn],0,1,0,0,0,...,None,None,None,None,None,None,None,None,None,None
4,5,"[{'amenity': 'Diện tích phòng: 35 m²', 'type':...",1 giường đôi lớn hoặc 2 giường đơn,"[{'bed_type': 'giường đôi lớn', 'bed_count': 1...","[giường đôi lớn, giường đơn]",1,0,1,0,0,...,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12042,11710,[],None,[],[],0,0,0,0,0,...,None,None,None,None,None,None,None,None,None,None
12043,11711,[],None,[],[],0,0,0,0,0,...,None,None,None,None,None,None,None,None,None,None
12044,11718,[],None,[],[],0,0,0,0,0,...,None,None,None,None,None,None,None,None,None,None
12045,11759,[],None,[],[],0,0,0,0,0,...,None,None,None,None,None,None,None,None,None,None


In [80]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12047 entries, 0 to 12046
Data columns (total 46 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   room_type_id                  12047 non-null  int64  
 1   room_amenities                12047 non-null  object 
 2   bed_str                       11377 non-null  object 
 3   expanded_bed                  12047 non-null  object 
 4   bed_list                      12047 non-null  object 
 5   large_double_bed              12047 non-null  int64  
 6   large_bed                     12047 non-null  int64  
 7   single_bed                    12047 non-null  int64  
 8   sofa_bed                      12047 non-null  int64  
 9   double_bed                    12047 non-null  int64  
 10  small_double_bed              12047 non-null  int64  
 11  king_size_bed                 12047 non-null  int64  
 12  futon_mattress                12047 non-null  int64  
 13  b

In [81]:
# df.to_csv("checkpoint7.csv", index=False)

## Handling bedroom

In [82]:
import ast
def extract_bedroom_features(amenities):
    if isinstance(amenities, str):
        amenities = ast.literal_eval(amenities)
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "bedroom":
           return amenity.get("amenity")
    return None

df["bedroom_str"] = df["room_amenities"].apply(extract_bedroom_features)
df[["bedroom_str"]].drop_duplicates()

,bedroom_str
0,None
50,2 phòng ngủ
92,3 phòng ngủ
292,Studio/1 phòng ngủ
331,12 phòng ngủ
338,4 phòng ngủ
849,7 phòng ngủ
855,13 phòng ngủ
911,5 phòng ngủ
913,15 phòng ngủ


In [83]:
import re

def extract_bedroom_count(text):
    if not isinstance(text, str):
        return 0
    # Ưu tiên tìm số ở đầu chuỗi (ví dụ: "2 phòng ngủ", "3 phòng ngủ")
    match = re.match(r"(\d+)", text)
    if match:
        return int(match.group(1))
    # Trường hợp đặc biệt: "Studio/1 phòng ngủ" hoặc "Studio"
    if "studio" in text.lower():
        return 1
    return 0

df["bedroom_count"] = df["bedroom_str"].apply(extract_bedroom_count)

In [84]:
df[["bedroom_count","bedroom_str"]].drop_duplicates()

,bedroom_count,bedroom_str
0,0,None
50,2,2 phòng ngủ
92,3,3 phòng ngủ
292,1,Studio/1 phòng ngủ
331,12,12 phòng ngủ
338,4,4 phòng ngủ
849,7,7 phòng ngủ
855,13,13 phòng ngủ
911,5,5 phòng ngủ
913,15,15 phòng ngủ


In [85]:
df.drop(columns=["bedroom_str"])

,room_type_id,room_amenities,bed_str,expanded_bed,bed_list,large_double_bed,large_bed,single_bed,sofa_bed,double_bed,...,complimentary-instant-coffee,jacuzzi-bathtub,smoking-allowed,electric-blanket,free-welcome-drink,low-floor,complimentary-tea,coffee-tea-maker,hot-spring-access,bedroom_count
0,1,"[{'amenity': 'Diện tích phòng: 18 m²', 'type':...",1 giường đôi lớn,"[{'bed_type': 'giường đôi lớn', 'bed_count': 1}]",[giường đôi lớn],1,0,0,0,0,...,None,None,None,None,None,None,None,None,None,0
1,2,"[{'amenity': 'Diện tích phòng: 30 m²', 'type':...",1 giường lớn,"[{'bed_type': 'giường lớn', 'bed_count': 1}]",[giường lớn],0,1,0,0,0,...,None,None,None,None,None,True,None,None,None,0
2,3,"[{'amenity': 'Diện tích phòng: 45 m²', 'type':...",2 giường lớn,"[{'bed_type': 'giường lớn', 'bed_count': 2}]",[giường lớn],0,1,0,0,0,...,None,None,True,None,None,True,None,None,None,0
3,4,"[{'amenity': 'Diện tích phòng: 20 m²', 'type':...",1 giường lớn,"[{'bed_type': 'giường lớn', 'bed_count': 1}]",[giường lớn],0,1,0,0,0,...,None,None,None,None,None,None,None,None,None,0
4,5,"[{'amenity': 'Diện tích phòng: 35 m²', 'type':...",1 giường đôi lớn hoặc 2 giường đơn,"[{'bed_type': 'giường đôi lớn', 'bed_count': 1...","[giường đôi lớn, giường đơn]",1,0,1,0,0,...,None,None,None,None,None,None,None,None,None,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12042,11710,[],None,[],[],0,0,0,0,0,...,None,None,None,None,None,None,None,None,None,0
12043,11711,[],None,[],[],0,0,0,0,0,...,None,None,None,None,None,None,None,None,None,0
12044,11718,[],None,[],[],0,0,0,0,0,...,None,None,None,None,None,None,None,None,None,0
12045,11759,[],None,[],[],0,0,0,0,0,...,None,None,None,None,None,None,None,None,None,0


## Handling confirmation instance

In [86]:
import ast
def extract_cf_in_features(amenities):
    if isinstance(amenities, str):
        amenities = ast.literal_eval(amenities)
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "confirmation-instant":
           return amenity.get("amenity")
    return None

df["confirmation-instant_str"] = df["room_amenities"].apply(extract_cf_in_features)
df[["confirmation-instant_str"]].drop_duplicates()

,confirmation-instant_str
0,Điều hòa cá nhân
1,Cửa sổ
2,Hành lang ngoài
10,None
16,Cửa sổ có thể mở ra
18,Ấm nước điện
22,Đồ dùng cho giấc ngủ thoải mái
26,Rượu
87,Đồ gỗ ngoài trời
91,Trái cây/đồ ăn vặt


In [87]:
def normalize_cf_in(views_str):
    import unicodedata
    if isinstance(views_str, str):
        # Xóa "Hướng" và strip
        s = views_str.replace("Hướng", "").strip()
        # Chuyển sang không dấu
        s = unicodedata.normalize('NFKD', s)
        s = ''.join([c for c in s if not unicodedata.combining(c)])
        s = s.replace('Đ', 'D').replace('đ', 'd')
        return s.lower()
    return None
df["confirmation-instant"] = df["confirmation-instant_str"].apply(normalize_cf_in)

In [88]:
df[["confirmation-instant"]].drop_duplicates()

,confirmation-instant
0,dieu hoa ca nhan
1,cua so
2,hanh lang ngoai
10,None
16,cua so co the mo ra
18,am nuoc dien
22,do dung cho giac ngu thoai mai
26,ruou
87,do go ngoai troi
91,trai cay/do an vat


In [89]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12047 entries, 0 to 12046
Data columns (total 50 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   room_type_id                  12047 non-null  int64  
 1   room_amenities                12047 non-null  object 
 2   bed_str                       11377 non-null  object 
 3   expanded_bed                  12047 non-null  object 
 4   bed_list                      12047 non-null  object 
 5   large_double_bed              12047 non-null  int64  
 6   large_bed                     12047 non-null  int64  
 7   single_bed                    12047 non-null  int64  
 8   sofa_bed                      12047 non-null  int64  
 9   double_bed                    12047 non-null  int64  
 10  small_double_bed              12047 non-null  int64  
 11  king_size_bed                 12047 non-null  int64  
 12  futon_mattress                12047 non-null  int64  
 13  b

## Export

In [90]:
df.drop(columns=["room_amenities", "bed_str", "bed_list","expanded_bed", "views_str", "bedroom_str", "confirmation-instant_str"]).to_csv("room_amenities_processed.csv")